In [ ]:
# Montar Drive e instalar dependencias

from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q

import torch
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ No GPU")

In [ ]:
# Descomprimir dataset desde Drive

import os, zipfile, glob

ZIP_PATH     = '/content/drive/MyDrive/mechdog_proyecto/dataset/mechdog-detector.zip'
EXTRACT_PATH = '/content/dataset'

os.makedirs(EXTRACT_PATH, exist_ok=True)

print("Descomprimiendo dataset...")
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_PATH)
print("✅ Dataset listo en:", EXTRACT_PATH)

yaml_files = glob.glob(f'{EXTRACT_PATH}/**/*.yaml', recursive=True)
YAML_PATH  = yaml_files[0]
print("YAML encontrado:", YAML_PATH)

In [ ]:
# Crear splits valid y test (el ZIP de Roboflow solo contiene train/)

import shutil, random

random.seed(42)

TRAIN_IMG = f'{EXTRACT_PATH}/train/images'
TRAIN_LBL = f'{EXTRACT_PATH}/train/labels'
VALID_IMG = f'{EXTRACT_PATH}/valid/images'
VALID_LBL = f'{EXTRACT_PATH}/valid/labels'
TEST_IMG  = f'{EXTRACT_PATH}/test/images'
TEST_LBL  = f'{EXTRACT_PATH}/test/labels'

for p in [VALID_IMG, VALID_LBL, TEST_IMG, TEST_LBL]:
    os.makedirs(p, exist_ok=True)

imagenes = [f for f in os.listdir(TRAIN_IMG) if f.endswith(('.jpg','.png'))]
random.shuffle(imagenes)

n_valid = int(len(imagenes) * 0.15)
n_test  = int(len(imagenes) * 0.15)

def mover(lista, src_img, src_lbl, dst_img, dst_lbl):
    for img_file in lista:
        shutil.move(os.path.join(src_img, img_file),
                    os.path.join(dst_img, img_file))
        lbl = os.path.splitext(img_file)[0] + '.txt'
        lbl_src = os.path.join(src_lbl, lbl)
        if os.path.exists(lbl_src):
            shutil.move(lbl_src, os.path.join(dst_lbl, lbl))

mover(imagenes[:n_valid],          TRAIN_IMG, TRAIN_LBL, VALID_IMG, VALID_LBL)
mover(imagenes[n_valid:n_valid+n_test], TRAIN_IMG, TRAIN_LBL, TEST_IMG,  TEST_LBL)

print("✅ Splits creados:")
for split, ruta in [('train', TRAIN_IMG), ('valid', VALID_IMG), ('test', TEST_IMG)]:
    n = len([f for f in os.listdir(ruta) if f.endswith(('.jpg','.png'))])
    print(f"  {split:<8}: {n} imágenes")

In [ ]:
# Corregir rutas del data.yaml

import yaml

with open(YAML_PATH, 'r') as f:
    config = yaml.safe_load(f)

config['path']  = EXTRACT_PATH
config['train'] = 'train/images'
config['val']   = 'valid/images'
config['test']  = 'test/images'

with open(YAML_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("✅ data.yaml actualizado")
print(f"\nClases ({config['nc']}):")
nombres = config['names']
if isinstance(nombres, list):
    for i, n in enumerate(nombres): print(f"  {i}: {n}")
else:
    for i, n in nombres.items():    print(f"  {i}: {n}")

In [ ]:
# ENTRENAMIENTO YOLOv8n

model = YOLO('yolov8n.pt')   # descarga pesos COCO base

results = model.train(
    data       = YAML_PATH,
    epochs     = 100,
    imgsz      = 640,
    batch      = 16,
    patience   = 20,
    lr0        = 0.01,
    device     = 0,
    project    = '/content/drive/MyDrive/mechdog_proyecto/modelos',
    name       = 'v1_entrenamiento',
    save       = True,
    save_period= 10,
)

print("✅ Entrenamiento completado")
print("Modelo guardado en:", results.save_dir)

In [ ]:
# Métricas sobre el conjunto de test
import numpy as np

model_best = YOLO(
    '/content/drive/MyDrive/mechdog_proyecto/'
    'modelos/v1_entrenamiento/weights/best.pt'
)

metrics = model_best.val(
    data  = YAML_PATH,
    split = 'test',
    conf  = 0.5,
)

nombres = list(metrics.names.values())

print("\n📊 MÉTRICAS SOBRE EL CONJUNTO DE PRUEBA")
print(f"  Precision    : {metrics.box.p.mean():.3f}")
print(f"  Recall       : {metrics.box.r.mean():.3f}")
print(f"  mAP@0.5      : {metrics.box.map50:.3f}")
print(f"  mAP@0.5:0.95 : {metrics.box.map:.3f}")

print("\nPor clase:")
for i, nombre in enumerate(nombres):
    print(f"  {nombre:<20} P:{metrics.box.p[i]:.3f}  "
          f"R:{metrics.box.r[i]:.3f}  AP50:{metrics.box.ap50[i]:.3f}")

In [ ]:
# Matriz de confusión (estilo Edge Impulse)

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

cm_raw    = metrics.confusion_matrix.matrix
etiquetas = nombres + ['background']
n         = len(etiquetas)

cm_norm = np.zeros_like(cm_raw, dtype=float)
for i in range(n):
    total = cm_raw[i].sum()
    if total > 0:
        cm_norm[i] = cm_raw[i] / total * 100

fig, ax = plt.subplots(figsize=(12, 8))
cmap = mcolors.LinearSegmentedColormap.from_list(
    'ei_style', ['#FFEBEE','#FFFFFF','#E8F5E9','#2E7D32'], N=256
)
im = ax.imshow(cm_norm, cmap=cmap, vmin=0, vmax=100, aspect='auto')
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(etiquetas, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(etiquetas, fontsize=9)
ax.set_xlabel('Predicho', fontsize=11, labelpad=10)
ax.set_ylabel('Real',     fontsize=11, labelpad=10)
ax.set_title('Confusion Matrix — YOLOv8n (test set)', fontsize=13, pad=15)

for i in range(n):
    for j in range(n):
        val   = cm_norm[i, j]
        texto = f'{val:.1f}%' if val > 0 else '0%'
        color = 'white' if val > 60 else ('#CCCCCC' if val == 0 else 'black')
        ax.text(j, i, texto, ha='center', va='center', fontsize=8,
                color=color, fontweight='bold' if i==j else 'normal')

plt.colorbar(im, ax=ax, label='%', shrink=0.8)
plt.tight_layout()

ruta_fig = '/content/drive/MyDrive/mechdog_proyecto/resultados/confusion_matrix_yolov8.png'
os.makedirs(os.path.dirname(ruta_fig), exist_ok=True)
plt.savefig(ruta_fig, dpi=150, bbox_inches='tight')
plt.show()
print("✅ Guardada en:", ruta_fig)

In [ ]:
# Tabla comparativa YOLOv8 vs FOMO

fomo_f1 = {
    'bolsa_ok':       0.81,
    'cables_ok':      0.83,
    'caja_ok':        0.85,
    'mesa_riesgo':    0.57,
    'mochila_ok':     0.87,
    'mochila_riesgo': 0.85,
    'silla_riesgo':   0.62,
}

yolo_ap50 = {nombre: float(metrics.box.ap50[i])
             for i, nombre in enumerate(metrics.names.values())}

avg_ap   = np.mean(list(yolo_ap50.values()))
avg_fomo = np.mean(list(fomo_f1.values()))

print("\n" + "="*55)
print(f"  {'Clase':<22} {'YOLOv8 AP50':>12} {'FOMO F1':>9}")
print("─"*55)
for clase in sorted(fomo_f1.keys()):
    print(f"  {clase:<22} {yolo_ap50.get(clase,0):>12.3f} "
          f"{fomo_f1[clase]:>9.3f}")
print("─"*55)
print(f"  {'PROMEDIO':<22} {avg_ap:>12.3f} {avg_fomo:>9.3f}")
print("="*55)
print(f"\n  Hardware YOLOv8 : Laptop RTX 4050")
print(f"  Hardware FOMO   : ESP32-CAM (~$8 USD)")
print(f"  Diferencia      : {(avg_ap - avg_fomo)*100:.1f} puntos porcentuales")
print(f"  Reducción costo : ~98%  ($600+ → $8 USD)")